In [ ]:
pip install tensorflow scikit-image pandas opencv-python tqdm

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Concatenate, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.preprocessing.image import img_to_array, load_img
from tqdm import tqdm
import cv2
from skimage.feature import local_binary_pattern

# Configuration
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 10
TRAIN_DIR = 'data/train'
TEST_DIR = 'data/test'
TRAIN_CSV = 'output_csv/train_biomarkers.csv'
TEST_CSV = 'output_csv/test_biomarkers.csv'

# --- Load Biomarkers ---
def load_biomarkers(train_csv, test_csv):
    train_df = pd.read_csv(train_csv).select_dtypes(include=[np.number])
    test_df = pd.read_csv(test_csv).select_dtypes(include=[np.number])
    X_train_bio = train_df.iloc[:, :-1].values.astype(np.float32)
    y_train = train_df.iloc[:, -1].values.astype(np.float32)
    X_test_bio = test_df.iloc[:, :-1].values.astype(np.float32)
    y_test = test_df.iloc[:, -1].values.astype(np.float32)
    return X_train_bio, y_train, X_test_bio, y_test

# --- LBP Computation ---
def compute_lbp_features(image_path):
    gray = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    resized = cv2.resize(gray, IMG_SIZE)
    lbp = local_binary_pattern(resized, P=8, R=1, method='uniform')
    hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, 59), range=(0, 58))
    return hist.astype(np.float32) / hist.sum()

# --- Image + LBP + Biomarker Generator ---
class HybridDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, image_dir, biomarkers, labels, batch_size):
        self.image_paths = []
        for cls in ['real', 'fake']:
            cls_dir = os.path.join(image_dir, cls)
            self.image_paths += [os.path.join(cls_dir, fname) for fname in sorted(os.listdir(cls_dir))]
        self.biomarkers = biomarkers
        self.labels = labels
        self.batch_size = batch_size

    def __len__(self):
        return len(self.image_paths) // self.batch_size

    def __getitem__(self, idx):
        batch_paths = self.image_paths[idx * self.batch_size:(idx + 1) * self.batch_size]
        imgs, lbps, bios, ys = [], [], [], []
        for i, path in enumerate(batch_paths):
            img = load_img(path, target_size=IMG_SIZE)
            img = img_to_array(img) / 255.0
            lbp = compute_lbp_features(path)
            label = 1.0 if 'fake' in path else 0.0
            bio = self.biomarkers[idx * self.batch_size + i]
            imgs.append(img)
            lbps.append(lbp)
            bios.append(bio)
            ys.append(label)
        return [np.array(imgs), np.array(lbps), np.array(bios)], np.array(ys)

# --- SSL ResNet Backbone (SimSiam-style pretraining stub) ---
def get_ssl_resnet_backbone():
    base = ResNet50(include_top=False, weights='imagenet', input_shape=(224, 224, 3))
    x = GlobalAveragePooling2D()(base.output)
    return Model(inputs=base.input, outputs=x)

# --- Build Hybrid Model ---
def build_hybrid_model():
    # Inputs
    img_input = Input(shape=(224, 224, 3), name="image_input")
    lbp_input = Input(shape=(58,), name="lbp_input")
    bio_input = Input(shape=(136,), name="bio_input")

    # Image CNN Backbone
    cnn_branch = get_ssl_resnet_backbone()(img_input)

    # LBP Dense Branch
    lbp_branch = Dense(64, activation='relu')(lbp_input)

    # Biomarker Branch
    bio_branch = Dense(64, activation='relu')(bio_input)

    # Concatenate all
    merged = Concatenate()([cnn_branch, lbp_branch, bio_branch])
    z = Dense(128, activation='relu')(merged)
    z = Dropout(0.4)(z)
    output = Dense(1, activation='sigmoid')(z)

    model = Model(inputs=[img_input, lbp_input, bio_input], outputs=output)
    model.compile(optimizer=Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy'])
    return model

# --- Training + Evaluation ---
X_train_bio, y_train, X_test_bio, y_test = load_biomarkers(TRAIN_CSV, TEST_CSV)

train_gen = HybridDataGenerator(TRAIN_DIR, X_train_bio, y_train, BATCH_SIZE)
test_gen = HybridDataGenerator(TEST_DIR, X_test_bio, y_test, BATCH_SIZE)

model = build_hybrid_model()
model.summary()

model.fit(train_gen, epochs=EPOCHS, validation_data=test_gen)

# --- Final Evaluation ---
y_true, y_pred = [], []
for X_batch, y_batch in tqdm(test_gen, desc="Testing"):
    preds = model.predict(X_batch)
    y_true.extend(y_batch)
    y_pred.extend((preds > 0.5).astype(int).flatten())

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))
